# Submission 07: Experiment 21G

This submission uses only the winning Experiment 21G configuration.

The model keeps the original numerical features, adds exact-value numerical target encoding, and keeps the original categorical features with one-hot encoding.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

PROJECT_ROOT = Path(r'C:\Users\aakif\Documents\DataCompetition')
TRAIN_PATH = PROJECT_ROOT / 'data' / 'train.csv'
TEST_PATH = PROJECT_ROOT / 'data' / 'test.csv'
SAMPLE_PATH = PROJECT_ROOT / 'data' / 'sample_submission.csv'
OUTPUT_PATH = PROJECT_ROOT / 'submissions' / 'submission_07.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

X_train = train.drop(columns=['Will_Buy_EV', 'id']).copy()
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})
X_test = test.drop(columns=['id']).copy()

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Numeric columns:', numeric_features)
print('Categorical columns:', categorical_features)


Training shape: (668665, 13)
Test shape: (286571, 13)
Numeric columns: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


In [2]:
def build_mapping(values, target, smoothing=20):
    temp = pd.DataFrame({
        'value': values.values,
        'target': target.values
    })

    global_mean = float(target.mean())
    stats = temp.groupby('value')['target'].agg(['mean', 'count'])

    smoothed = (
        stats['count'] * stats['mean']
        + smoothing * global_mean
    ) / (stats['count'] + smoothing)

    return smoothed, global_mean


def add_numeric_identity_target_encoding(X_tr, y_tr, X_te, columns, smoothing=20):
    train_out = X_tr.copy()
    test_out = X_te.copy()

    for col in columns:
        train_values = train_out[col].astype('string').fillna('__MISSING__')
        test_values = test_out[col].astype('string').fillna('__MISSING__')

        mapping, global_mean = build_mapping(
            train_values,
            y_tr,
            smoothing=smoothing
        )

        train_out[f'{col}__identity_target'] = (
            train_values
            .map(mapping)
            .fillna(global_mean)
            .astype(float)
        )

        test_out[f'{col}__identity_target'] = (
            test_values
            .map(mapping)
            .fillna(global_mean)
            .astype(float)
        )

    return train_out, test_out


X_train_encoded, X_test_encoded = add_numeric_identity_target_encoding(
    X_train,
    y,
    X_test,
    numeric_features,
    smoothing=20
)

encoded_columns = [
    c for c in X_train_encoded.columns
    if '__identity_target' in c
]

print('Original feature count:', X_train.shape[1])
print('Final feature count:', X_train_encoded.shape[1])
print('Identity target columns:', len(encoded_columns))
print('Identity target columns:', encoded_columns)

if len(encoded_columns) != len(numeric_features):
    raise ValueError('Not all numerical columns received identity target encoding.')


Original feature count: 13
Final feature count: 20
Identity target columns: 7
Identity target columns: ['Age__identity_target', 'Annual_Income_USD__identity_target', 'Daily_Commute_km__identity_target', 'Number_of_Cars_Owned__identity_target', 'Charging_Stations_Near_Home__identity_target', 'Charging_Stations_Near_Work__identity_target', 'Environmental_Concern_Level__identity_target']


In [3]:
numeric_model_features = X_train_encoded.select_dtypes(include=['number']).columns.tolist()
categorical_model_features = X_train_encoded.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer([
    (
        'num',
        Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]),
        numeric_model_features
    ),
    (
        'cat',
        Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]),
        categorical_model_features
    )
])

model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

print('Fitting final Submission 07 model...')
pipeline.fit(X_train_encoded, y)

print('Generating test predictions...')
test_predictions = pipeline.predict_proba(X_test_encoded)[:, 1]

print('Prediction count:', len(test_predictions))
print('Prediction minimum:', float(test_predictions.min()))
print('Prediction maximum:', float(test_predictions.max()))
print('Prediction mean:', float(test_predictions.mean()))


Fitting final Submission 07 model...
Generating test predictions...
Prediction count: 286571
Prediction minimum: 3.6679500681202626e-06
Prediction maximum: 0.9990504384040833
Prediction mean: 0.17314328253269196


In [4]:
target_column = 'Will_Buy_EV'

if target_column not in sample_submission.columns:
    raise ValueError('Will_Buy_EV column was not found in sample_submission.csv')

if len(sample_submission) != len(test_predictions):
    raise ValueError(
        f'Sample submission has {len(sample_submission)} rows, '
        f'but predictions have {len(test_predictions)} rows.'
    )

submission = sample_submission.copy()
submission[target_column] = test_predictions

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

assert OUTPUT_PATH.exists()
assert len(submission) == len(test)
assert submission[target_column].notna().all()
assert np.isfinite(submission[target_column]).all()
assert ((submission[target_column] >= 0) & (submission[target_column] <= 1)).all()

print('')
print('=' * 70)
print('SUBMISSION 07 READY')
print('=' * 70)
print('File:', OUTPUT_PATH)
print('Rows:', len(submission))
print('Columns:', submission.columns.tolist())
print('Prediction range:', float(submission[target_column].min()), 'to', float(submission[target_column].max()))
print('')
print(submission.head().to_string(index=False))
print('')
print('Submission validation passed.')



SUBMISSION 07 READY
File: C:\Users\aakif\Documents\DataCompetition\submissions\submission_07.csv
Rows: 286571
Columns: ['id', 'Will_Buy_EV']
Prediction range: 3.6679500681202626e-06 to 0.9990504384040833

    id  Will_Buy_EV
668665     0.027427
668666     0.013836
668667     0.002981
668668     0.003124
668669     0.029330

Submission validation passed.
